# Week05 Part 2 완성본 — 공개 사전학습 MobileNetV2 전이학습

## 실행 순서
Notion의 **Part 2 실습 1 → 8**과 코드 셀이 대응합니다.

Teachable Machine 모델이 없어도 이 노트북은 독립적으로 실행할 수 있습니다.
기본 모델은 ImageNet으로 사전학습된 **MobileNetV2**입니다.


## 실습 1. 환경 확인


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 실습 2. CIFAR-10에서 3개 클래스 데이터 준비


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

selected = [0, 1, 2]
class_names = ["airplane", "automobile", "bird"]

train_mask = np.isin(y_train, selected)
test_mask = np.isin(y_test, selected)

x_train = x_train[train_mask][:4500]
y_train = y_train[train_mask][:4500]
x_test = x_test[test_mask][:900]
y_test = y_test[test_mask][:900]

print("train:", x_train.shape, y_train.shape)
print("test :", x_test.shape, y_test.shape)


## 실습 3. 데이터 일부 확인


In [ ]:
plt.figure(figsize=(9, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[int(y_train[i])])
    plt.axis("off")
plt.tight_layout()
plt.show()


## 실습 4. MobileNetV2용 전처리와 Dataset 구성


In [ ]:
IMG_SIZE = 96
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = (
    train_ds
    .shuffle(2000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds = (
    test_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


## 실습 5. MobileNetV2 Feature Extraction 모델 생성


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(3, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.summary()


## 실습 6. Feature Extraction 학습


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3
)


## 실습 7. Fine-tuning


In [ ]:
base_model.trainable = True

fine_tune_at = len(base_model.layers) - 25
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=2
)


## 실습 8. 평가


In [ ]:
loss, acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {acc:.4f}")


## 실습 9. Feature Extraction / Fine-tuning 그래프 비교


In [ ]:
plt.figure(figsize=(8, 4))

feature_epochs = range(1, len(history.history["val_accuracy"]) + 1)
fine_epochs = range(
    len(history.history["val_accuracy"]) + 1,
    len(history.history["val_accuracy"]) + len(fine_history.history["val_accuracy"]) + 1
)

plt.plot(feature_epochs, history.history["val_accuracy"], marker="o", label="Feature Extraction")
plt.plot(fine_epochs, fine_history.history["val_accuracy"], marker="o", label="Fine-tuning")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.legend()
plt.show()


## 실습 10. 예측 결과 확인


In [ ]:
batch_images, batch_labels = next(iter(test_ds))
pred = model.predict(batch_images[:9], verbose=0)
pred_cls = np.argmax(pred, axis=1)

plt.figure(figsize=(9, 9))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    img = (batch_images[i].numpy() + 1.0) / 2.0
    plt.imshow(np.clip(img, 0, 1))
    plt.title(
        f"T: {class_names[int(batch_labels[i])]}\n"
        f"P: {class_names[int(pred_cls[i])]}"
    )
    plt.axis("off")
plt.tight_layout()
plt.show()


## 실습 11. 모델 저장


In [ ]:
model.save("Week05_Part2_MobileNetV2_3class.keras")
print("모델 저장 완료")


## 결과 분석

다음 표를 Notion 워크북에 기록하세요.

| 항목 | Feature Extraction | Fine-tuning |
|---|---|---|
| 학습 정확도 |  |  |
| 검증 정확도 |  |  |
| 학습 시간 |  |  |
| 특징 |  |  |

### 질문
1. Fine-tuning 후 정확도가 좋아졌는가?
2. 좋아지지 않았다면 어떤 이유가 가능한가?
3. `learning_rate=1e-5`처럼 작은 값을 사용하는 이유는?
4. `base_model.trainable=False`의 의미는?
